In [1]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats

In [2]:
from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
import sklearn.linear_model as linear_model
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [3]:
train = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/train.csv')
test = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv').drop(columns=['id'])

In [4]:
encoder = LabelEncoder()
normalize = SimpleImputer(strategy='mean')

In [5]:
for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]] = train[i[0]].fillna('Unknown')
        train[i[0]]=normalize.fit_transform(encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1)).reshape(-1,1))
    else:
        train[i[0]].fillna(train[i[0]].mean(), inplace=True)
        train[i[0]]=normalize.fit_transform(numpy.array(train[i[0]]).reshape(-1,1))
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]] = test[i[0]].fillna('Unknown')
        test[i[0]]=normalize.fit_transform(numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1))).reshape(-1,1))
    else:
        test[i[0]].fillna(test[i[0]].mean(), inplace=True)
        test[i[0]]=normalize.fit_transform(numpy.array(test[i[0]]).reshape(-1,1))

In [6]:
train_x = train.drop(columns=['id', 'efficiency'])
train_y = train['efficiency']

In [7]:
LGBM_R_parm = {'colsample_bytree': 0.6657998165699125,
               'drop_rate': 0.8303473371870002,
               'learning_rate': 0.2762739125344964,
               'max_bin': 9983,
               'max_depth': 8623,
               'max_drop': 5480,
               'min_child_samples': 101,
               'min_data_in_leaf': 426,
               'n_estimators': 1628,
               'num_leaves': 3640,
               'objective': 'regression_l1',
               'reg_alpha': 0.39940189926691194,
               'reg_lambda': 0.9567353299218986,
               'skip_drop': 0.6274640597528743,
               'verbosity': -1}

XGB_R_parm = {'colsample_bytree': 0.871893537724492,
              'gamma': 1,
              'learning_rate': 0.923192518624813,
              'max_depth': 15,
              'min_child_weight': 1,
              'n_estimators': 17748,
              'objective': 'binary:logistic',
              'reg_alpha': 45,
              'reg_lambda': 0.8598696247943665,
              'subsample': 0.9109367356405415}

catboost_params = {'iterations' : 1000,
                   'learning_rate': 0.009, 
                   'depth': 5, 
                   'l2_leaf_reg': 5.5,
                   'min_child_samples' : 102,
                   'od_wait' : 50,
                   'random_state' : 42,
                   'eval_metric': 'RMSE', 
                   'bootstrap_type': 'Bayesian', 
                   'grow_policy' : 'Depthwise',
                   'logging_level' : 'Silent'
                 }

X_train, X_test, y_train, y_test = train_test_split(train_x, train_y, test_size=0.1, random_state=100)

In [8]:
LGBM_R = LGBMRegressor(**LGBM_R_parm)

In [9]:
XGB_R = XGBRegressor(**XGB_R_parm)

In [10]:
catboost = CatBoostRegressor(learning_rate=0.009,
                             depth=5,
                             l2_leaf_reg=5.5,
                             min_child_samples=102,
                             grow_policy='Depthwise',
                             iterations=1000,
                             eval_metric='RMSE',
                             od_type='Iter',
                             od_wait=50,
                             random_state=42,
                             logging_level='Silent')

In [11]:
estimators = [
    ('LGBM', make_pipeline(MinMaxScaler(), LGBM_R)),
    ('xgb', make_pipeline(MinMaxScaler(), XGB_R)),
    ('cat', make_pipeline(MinMaxScaler(), catboost))
]

model = StackingRegressor(estimators, final_estimator = RidgeCV(alphas=numpy.logspace(-3,10, 20)))
model.fit(X_train, y_train)

StackingRegressor(estimators=[('LGBM',
                               Pipeline(steps=[('minmaxscaler', MinMaxScaler()),
                                               ('lgbmregressor',
                                                LGBMRegressor(colsample_bytree=0.6657998165699125,
                                                              drop_rate=0.8303473371870002,
                                                              learning_rate=0.2762739125344964,
                                                              max_bin=9983,
                                                              max_depth=8623,
                                                              max_drop=5480,
                                                              min_child_samples=101,
                                                              min_data_in_leaf=426,
                                                              n_estimators=1628,
                                                              num_leaves=3640,
                                                              objective='regre...
                                                <catboost.core.CatBoostRegressor object at 0x78062ed4d190>)]))],
                  final_estimator=RidgeCV(alphas=array([1.00000000e-03, 4.83293024e-03, 2.33572147e-02, 1.12883789e-01,
       5.45559478e-01, 2.63665090e+00, 1.27427499e+01, 6.15848211e+01,
       2.97635144e+02, 1.43844989e+03, 6.95192796e+03, 3.35981829e+04,
       1.62377674e+05, 7.84759970e+05, 3.79269019e+06, 1.83298071e+07,
       8.85866790e+07, 4.28133240e+08, 2.06913808e+09, 1.00000000e+10])))

In [12]:
print(f'Concordance index (lifelines): {100*(1-numpy.sqrt(mean_squared_error(y_test, model.predict(X_test))))}')

Concordance index (lifelines): 90.26802648099188


In [13]:
id = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv')['id']
test_predictions = model.predict(test)
test_predictions

array([0.39994171, 0.54046453, 0.52350402, ..., 0.62395322, 0.42998355,
       0.56135528])

In [14]:
submission = pandas.DataFrame({
    'id': id.values,
    'efficiency': test_predictions,
})
submission

,id,efficiency
0,0,0.399942
1,1,0.540465
2,2,0.523504
3,3,0.471282
4,4,0.459329
...,...,...
11995,11995,0.556365
11996,11996,0.460220
11997,11997,0.623953
11998,11998,0.429984


In [15]:
submission.to_csv('submission.csv', index = False)